### Add Flight Names Column

In [6]:
import csv

input_file_path = "data/dataset_original.csv"
output_file_path = "data/dataset_flights.csv"

with open(input_file_path, "r", encoding="utf-8") as fin, open(
    output_file_path, "w", encoding="utf-8", newline=""
) as fout:

    reader = csv.reader(fin)
    writer = csv.writer(fout)

    # Write a header including the Label column
    writer.writerow(["Request", "Flight", "Label"])

    # Skip the header row of the input file if it exists
    input_header = next(reader, None)
    if input_header and input_header[0].strip().lower() == "request":
        pass  # Skip the header row

    for row in reader:
        # Each row is [request, label], so the request is row[0] and label is row[1]
        request_text = row[0]
        label = row[1] if len(row) > 1 else ""

        # Extract flight name by splitting on commas
        parts = request_text.split(",")
        flight = parts[1].strip() if len(parts) > 1 else ""

        # Write request, flight, and label to the new CSV
        writer.writerow([request_text, flight, label])

### Remove Punctuation and Lowercase

In [7]:
import pandas as pd
import string

# Load the dataset
file_path = "data/dataset_flights.csv"  # Change this to your actual dataset file path
df = pd.read_csv(file_path)

# Function to remove punctuation
def remove_punctuation(text):
    return (
        text.translate(str.maketrans("", "", string.punctuation))
        if isinstance(text, str)
        else text
    )

# Apply function only to 'Request' column if it exists
if "Request" in df.columns:
    df["Request"] = df["Request"].apply(remove_punctuation)

# Save to a new file
new_file_path = "data/dataset_no_punctuation.csv"
df.to_csv(new_file_path, index=False)

print(f"Processed dataset saved as {new_file_path}")

Processed dataset saved as data/dataset_no_punctuation.csv


### Change Numbers to Words

In [8]:
import pandas as pd
import re
from num2words import num2words

# Load the dataset
file_path = "data/dataset_no_punctuation.csv"  # Ensure this is the correct file
df = pd.read_csv(file_path)


# Function to replace numbers with words and format runway identifiers
def numbers_to_words(text):
    if isinstance(text, str):
        # Convert runway identifiers first (e.g., 18R -> one eight right)
        text = re.sub(
            r"runway (\d+)([LRC])",
            lambda x: f"runway {' '.join(num2words(int(digit)) for digit in x.group(1))} "
            f"{'left' if x.group(2) == 'L' else 'right' if x.group(2) == 'R' else 'center'}",
            text,
            flags=re.IGNORECASE,
        )

        # Convert standalone numbers to words (excluding those already handled in runways)
        text = re.sub(
            r"\b\d+\b",
            lambda x: " ".join(num2words(int(digit)) for digit in x.group()),
            text,
        )

        return text
    return text


# Apply function only to 'Request' column if it exists
if "Request" in df.columns:
    df["Request"] = df["Request"].apply(numbers_to_words)

# Save to a new file
new_file_path = "data/dataset_words_only.csv"
df.to_csv(new_file_path, index=False)

print(f"Processed dataset saved as {new_file_path}")

Processed dataset saved as data/dataset_words_only.csv


### Add Homophones

In [ ]:
import pandas as pd
import random
import re
import sys
import time
from utils.homophones import get_homophones
from utils.rhymes import get_rhymes

homophone_cache = {}
rhyme_cache = {}

COMMON_WORDS = {
    "the",
    "and",
    "is",
    "in",
    "on",
    "to",
    "of",
    "for",
    "it",
    "a",
    "an",
    "at",
}

AVIATION_TERMS = {
    "takeoff": ["take", "off"],
    "pushback": ["push", "back"],
    "taxiway": ["taxi", "way"],
    "runway": ["run", "way"],
    "airline": ["air", "line"],
    "touchdown": ["touch", "down"],
    "flyover": ["fly", "over"],
}


def fetch_homophones(word):
    if word in homophone_cache:
        return homophone_cache[word]

    homophones = get_homophones(word)
    homophone_cache[word] = homophones
    return homophones


def fetch_rhymes(word):
    if word in rhyme_cache:
        return rhyme_cache[word]

    rhymes = get_rhymes(word)
    rhyme_cache[word] = rhymes
    return rhymes


def split_word_if_needed(word):
    if word.lower() in AVIATION_TERMS:
        return AVIATION_TERMS[word.lower()]
    return [word]


def substitute_word(word, homophone_prob, rhyme_prob, split_prob):
    clean_word = re.sub(r"[^\w\s]", "", word)
    if clean_word.lower() in COMMON_WORDS:
        return word

    rand_val = random.random()

    if rand_val < homophone_prob:
        homophones = fetch_homophones(clean_word.lower())
        homophones = [h for h in homophones if h.lower() != clean_word.lower()]
        if homophones:
            new_word = random.choice(homophones)
            return new_word.capitalize() if word.istitle() else new_word

    elif rand_val < homophone_prob + rhyme_prob:
        rhymes = fetch_rhymes(clean_word.lower())
        rhymes = [r for r in rhymes if r.lower() != clean_word.lower()]
        if rhymes:
            new_word = random.choice(rhymes)
            return new_word.capitalize() if word.istitle() else new_word

    elif rand_val < homophone_prob + rhyme_prob + split_prob:
        split_words = split_word_if_needed(clean_word.lower())
        return (
            " ".join(split_words).capitalize()
            if word.istitle()
            else " ".join(split_words)
        )

    return word


def substitute_with_homophones_and_rhymes(text, homophone_prob, rhyme_prob, split_prob):
    if not isinstance(text, str):
        return text

    words = text.split()
    new_words = [
        substitute_word(word, homophone_prob, rhyme_prob, split_prob) for word in words
    ]
    return " ".join(new_words)


# Load the dataset
file_path = "data/dataset_words_only.csv"
df = pd.read_csv(file_path)

# Duplicate the dataset to double its size
df_augmented = pd.concat([df, df.copy()], ignore_index=True)

total_rows = len(df_augmented)
start_time = time.time()

# Apply augmentation **only to the duplicated rows**
if "Request" in df_augmented.columns:
    for i in range(len(df), total_rows):  # Only augment the new copies
        df_augmented.at[i, "Request"] = substitute_with_homophones_and_rhymes(
            df_augmented.at[i, "Request"],
            homophone_prob=0.5,
            rhyme_prob=0.2,
            split_prob=0.6,
        )

        # Live progress update
        progress = f"Processing row {i - len(df) + 1}/{total_rows - len(df)} ({((i - len(df) + 1) / (total_rows - len(df))) * 100:.2f}%)"
        sys.stdout.write("\r" + progress)
        sys.stdout.flush()

# Save the augmented dataset
new_file_path = "data/dataset_augmented.csv"
df_augmented.to_csv(new_file_path, index=False)

elapsed_time = time.time() - start_time
print(
    f"\nDataset with homophones, rhymes, and word splits saved as {new_file_path} (Completed in {elapsed_time:.2f} seconds)"
)

Processing row 396/396 (100.00%)
Dataset with homophones, rhymes, and word splits saved as data/dataset_augmented.csv (Completed in 227.49 seconds)


### Add Candidates

In [15]:
import pandas as pd
import random

# Load dataset
file_path = "data/dataset_augmented.csv"
df = pd.read_csv(file_path)

# Extract unique airline names from dataset
all_airlines = list(set([" ".join(flight.split()[:-1]) for flight in df["Flight"]]))


# Function to extract airline name from flight identifier
def extract_airline(flight):
    return " ".join(flight.split()[:-1])  # Extract all but last part as airline name


# Function to generate a completely random flight number for the same airline
def generate_random_flight_number(correct_flight):
    airline = extract_airline(correct_flight)  # Extract airline name
    random_number = random.randint(100, 999)  # Fully random flight number
    return f"{airline} {random_number}"


# Function to generate random incorrect candidates
def generate_random_candidates(correct_flight):
    correct_airline = extract_airline(correct_flight)

    # Ensure the correct flight is in the list
    candidates = [correct_flight]

    # Add one completely random flight number from the same airline
    candidates.append(generate_random_flight_number(correct_flight))

    # 20% chance of adding another random flight number from the same airline
    if random.random() < 0.2:
        candidates.append(generate_random_flight_number(correct_flight))

    # Generate remaining incorrect flights from different airlines
    incorrect_candidates = []
    for _ in range(5 - len(candidates)):  # Fill up to 5 candidates
        random_airline = random.choice(
            [a for a in all_airlines if a != correct_airline]
        )
        random_number = random.randint(100, 999)
        incorrect_candidates.append(f"{random_airline} {random_number}")

    candidates.extend(incorrect_candidates)  # Add incorrect candidates
    random.shuffle(candidates)  # Shuffle to avoid positional bias

    return candidates


# Generate candidate flights for each request
df["Candidates"] = df["Flight"].apply(generate_random_candidates)

# Save the new dataset
output_file = "data/dataset_final.csv"
df.to_csv(output_file, index=False)

print(f"✅ Dataset saved as {output_file}")

✅ Dataset saved as data/dataset_final.csv


### Change Flight Names to Words

In [16]:
import pandas as pd
from num2words import num2words
import ast

# Load dataset
file_path = "data/dataset_final.csv"
df = pd.read_csv(file_path)


# Function to convert flight numbers to words
def convert_flight_to_words(flight):
    parts = flight.split()
    airline = " ".join(parts[:-1])  # Extract airline name
    number = parts[-1]  # Extract flight number
    number_in_words = " ".join(
        num2words(int(digit)) for digit in number
    )  # Convert to words
    return f"{airline} {number_in_words}"


# Apply the conversion to the Flight column
df["Flight"] = df["Flight"].apply(convert_flight_to_words)


# Convert Candidates column (which is stored as a string list)
def convert_candidates(candidates):
    candidates = ast.literal_eval(candidates)  # Convert string list to actual list
    return [convert_flight_to_words(flight) for flight in candidates]


df["Candidates"] = df["Candidates"].apply(convert_candidates)

# Save the new dataset
output_file = "data/dataset_final.csv"
df.to_csv(output_file, index=False)

print(f"✅ Dataset saved as {output_file}")

✅ Dataset saved as data/dataset_final.csv


### Split Dataset to Train and Test

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
file_path = "data/dataset_final.csv"
df = pd.read_csv(file_path)

# Split the dataset into 90% training and 10% testing
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, shuffle=True)

# Save the split datasets
train_df.to_csv("data/dataset_final_train.csv", index=False)
test_df.to_csv("data/dataset_final_test.csv", index=False)

print("✅ Dataset split complete! Train and test files saved.")